# Alexa Dandridge - Initial EDA for Kaggle Competition (4/15/2026)

In [20]:
# Importing Libraries
import pandas as pd
import numpy as np
import sweetviz as sv
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.ensemble import RandomForestClassifier #bagging model
from xgboost import XGBClassifier #boosting model
from sklearn.metrics import classification_report, accuracy_score

In [21]:
# importing training data
train = pd.read_csv('train.csv')
train.head()


,id,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,...,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region,Irrigation_Need
0,0,Loamy,4.92,32.58,1.01,3.05,15.01,50.61,725.99,5.90,...,Sugarcane,Sowing,Zaid,Drip,Rainwater,0.82,No,112.16,East,Low
1,1,Clay,7.08,56.61,0.44,2.00,22.92,67.86,985.66,6.98,...,Wheat,Vegetative,Kharif,Rainfed,River,5.27,Yes,47.16,South,Low
2,2,Clay,5.69,27.71,0.81,2.83,26.97,92.22,2201.70,6.05,...,Rice,Vegetative,Kharif,Sprinkler,Reservoir,8.24,Yes,110.38,North,Low
3,3,Sandy,5.65,13.32,1.33,0.87,13.32,61.57,1357.33,9.12,...,Wheat,Flowering,Kharif,Canal,River,8.32,Yes,53.85,South,Medium
4,4,Clay,7.96,59.14,0.38,0.96,20.22,91.11,1538.20,6.95,...,Wheat,Sowing,Rabi,Canal,River,7.37,No,93.19,South,Low


In [22]:
# importing testing data
test = pd.read_csv('test.csv') 
test.head()

,id,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,Wind_Speed_kmh,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region
0,630000,Silt,6.36,26.19,0.59,2.81,17.83,30.24,1533.38,5.40,3.00,Maize,Sowing,Rabi,Canal,River,13.59,Yes,47.48,West
1,630001,Clay,5.87,9.88,1.18,3.26,21.18,78.07,576.05,7.22,15.88,Cotton,Sowing,Rabi,Drip,Reservoir,6.12,Yes,56.43,South
2,630002,Sandy,6.22,26.55,0.96,0.85,26.87,60.35,545.30,9.43,2.63,Wheat,Sowing,Kharif,Sprinkler,Reservoir,3.11,Yes,20.00,East
3,630003,Clay,7.68,53.58,0.83,0.55,41.74,36.05,1211.03,6.69,1.86,Maize,Harvest,Rabi,Canal,Groundwater,2.27,No,102.99,North
4,630004,Loamy,5.23,59.02,0.54,2.11,41.08,52.47,1321.91,4.11,5.71,Cotton,Sowing,Kharif,Canal,Groundwater,12.39,Yes,13.33,Central


In [23]:
# Finding the basic structure of the data
print(train.shape)
print(test.shape)


(630000, 21)
(270000, 20)


In [24]:
# Finding missing values - no missing values in training data
print(train.isnull().sum())

id                         0
Soil_Type                  0
Soil_pH                    0
Soil_Moisture              0
Organic_Carbon             0
Electrical_Conductivity    0
Temperature_C              0
Humidity                   0
Rainfall_mm                0
Sunlight_Hours             0
Wind_Speed_kmh             0
Crop_Type                  0
Crop_Growth_Stage          0
Season                     0
Irrigation_Type            0
Water_Source               0
Field_Area_hectare         0
Mulching_Used              0
Previous_Irrigation_mm     0
Region                     0
Irrigation_Need            0
dtype: int64


In [25]:
# Using SweetViz to analyze the data
report = sv.analyze(train)
report.show_html('train_report.html')

                                             |          | [  0%]   00:00 -> (? left)

Exception ignored in: <function ResourceTracker.__del__ at 0x102f99800>
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 84, in __del__
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 93, in _stop
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 118, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x102f35800>
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 84, in __del__
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 93, in _stop
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 118, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x10513d800>
Traceback (most recent call last

Report train_report.html was generated! NOTEBOOK/COLAB USERS: the web browser MAY not pop up, regardless, the report IS saved in your notebook/colab files.


Usings two distinct modeling approaches, one bagging and one boosting.

In [26]:
# Dropping ID
train = train.drop('id', axis=1)
test_ids = test['id']
test = test.drop('id', axis=1)

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import balanced_accuracy_score

# Separating features and target variable
X = train.drop('Irrigation_Need', axis=1)

# Encode target labels for XGBoost
le = LabelEncoder()
y = le.fit_transform(train['Irrigation_Need'])

print(le.classes_)

# Encoding categorical variables
X = pd.get_dummies(X)
test = pd.get_dummies(test)

# Align columns
X, test = X.align(test, join='left', axis=1, fill_value=0)

# Train/test split
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

['High' 'Low' 'Medium']


In [27]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import balanced_accuracy_score

rf = RandomForestClassifier(
    n_estimators=50,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)

skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

rf_cv_scores = cross_val_score(rf, X, y, cv=skf, scoring='balanced_accuracy', n_jobs=-1)

print("Random Forest CV Scores:", rf_cv_scores)
print("Random Forest Mean CV:", rf_cv_scores.mean())

rf.fit(X_train, y_train)
rf_pred = rf.predict(X_val)

print("Random Forest Validation Score:", balanced_accuracy_score(y_val, rf_pred))

Random Forest CV Scores: [0.87272126 0.90197488 0.87831561]
Random Forest Mean CV: 0.8843372500237866
Random Forest Validation Score: 0.8895625252639004


In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import balanced_accuracy_score

xgb = XGBClassifier(
    n_estimators=100,        
    max_depth=4,             
    learning_rate=0.1,
    subsample=0.8,              
    random_state=42,
    n_jobs=-1
)

# cross-validation
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

xgb_cv_scores = cross_val_score(
    xgb, X, y,
    cv=skf,
    scoring='balanced_accuracy',
    n_jobs=-1
)

print("XGBoost CV Scores:", xgb_cv_scores)
print("XGBoost Mean CV:", xgb_cv_scores.mean())

# validation performance
xgb.fit(X_train, y_train)
xgb_pred = xgb.predict(X_val)

print("XGBoost Validation Score:", balanced_accuracy_score(y_val, xgb_pred))

XGBoost CV Scores: [0.95846951 0.95846058 0.95925208]
XGBoost Mean CV: 0.958727389105102
XGBoost Validation Score: 0.9623907297547464


Exception ignored in: <function ResourceTracker.__del__ at 0x102ed9800>
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 84, in __del__
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 93, in _stop
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 118, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x104d69800>
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 84, in __del__
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 93, in _stop
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 118, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x107099800>
Traceback (most recent call last

In [29]:
rf.fit(X, y)
rf_test_preds = rf.predict(test)
rf_test_labels = le.inverse_transform(rf_test_preds)

rf_submission = pd.DataFrame({
    'id': test_ids,
    'Irrigation_Need': rf_test_labels
})

rf_submission.to_csv('rf_submission.csv', index=False)

In [ ]:
xgb.fit(X, y)
xgb_test_preds = xgb.predict(test)
xgb_test_labels = le.inverse_transform(xgb_test_preds)

xgb_submission = pd.DataFrame({
    'id': test_ids,
    'Irrigation_Need': xgb_test_labels
})

xgb_submission.to_csv('xgb_submission.csv', index=False)

Exception ignored in: <function ResourceTracker.__del__ at 0x1064d5800>
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 84, in __del__
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 93, in _stop
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 118, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x103771800>
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 84, in __del__
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 93, in _stop
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 118, in _stop_locked
ChildProcessError: [Errno 10] No child processes
